In [1]:
import os
import random
from PIL import Image, ImageOps, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
WORDS_FILE = 'iam_data/words_new.txt'
IMAGE_DIR = 'iam_data/iam_words/words'

In [3]:
characters = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789.,!?'-\"():;#&/ "
char_to_idx = {c: i + 1 for i, c in enumerate(characters)}
idx_to_char = {i + 1: c for i, c in enumerate(characters)}
BLANK_IDX = 0

In [4]:
class WordDataset(Dataset):
    def __init__(self, txt_file, img_dir, max_samples=None):
        self.img_dir = img_dir
        self.data = []

        with open(txt_file, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        for line in lines:
            line = line.strip()

            if len(line) == 0 or line.startswith('#'):
                continue

            parts = line.split()

            if len(parts) < 9:
                continue

            word_id = parts[0]
            status = parts[1]

            if status != 'ok':
                continue

            text = parts[-1]

            valid = True
            encoded = []
            for ch in text:
                if ch not in char_to_idx:
                    valid = False
                    break
                encoded.append(char_to_idx[ch])

            if not valid:
                continue

            filename = word_id + '.png'
            self.data.append((filename, encoded))

        if max_samples is not None:
            self.data = self.data[:max_samples]

        print(f'Total valid samples: {len(self.data)}')

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        filename, encoded_text = self.data[idx]

        word_id = filename.replace('.png', '')
        parts = word_id.split('-')

        folder1 = parts[0]
        folder2 = parts[0] + '-' + parts[1]

        img_path = os.path.join(
            self.img_dir,
            folder1,
            folder2,
            filename
        )

        try:
            img = Image.open(img_path).convert('L')
        except (FileNotFoundError, UnidentifiedImageError, OSError):
            return self.__getitem__((idx + 1) % len(self.data))

        # Resize height to 32 and keep aspect ratio
        new_w = int(img.size[0] * (32 / img.size[1]))
        img = img.resize((new_w, 32), Image.LANCZOS)

        # Pad / crop width to 128
        if new_w < 128:
            img = ImageOps.expand(img, border=(0, 0, 128 - new_w, 0), fill=255)
        else:
            img = img.crop((0, 0, 128, 32))

        img = self.transform(img)
        target = torch.tensor(encoded_text, dtype=torch.long)

        return img, target

In [5]:
def collate_fn(batch):
    images, labels = zip(*batch)

    images = torch.stack(images)

    target_lengths = torch.tensor(
        [len(label) for label in labels],
        dtype=torch.long
    )

    targets = torch.cat(labels)

    return images, targets, target_lengths

In [6]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 1))
        )

        self.rnn = nn.LSTM(
            input_size=256 * 4,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        # x: [B,1,32,128]
        x = self.cnn(x)
        # x: [B,256,4,32]

        b, c, h, w = x.size()

        x = x.permute(0, 3, 1, 2)
        x = x.contiguous().view(b, w, c * h)

        x, _ = self.rnn(x)
        x = self.fc(x)

        return x  # [B,T,C]

In [7]:
word_train_ds = WordDataset(
    WORDS_FILE,
    IMAGE_DIR,
    max_samples=5000
)

random.shuffle(word_train_ds.data)

train_loader = DataLoader(
    word_train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

Total valid samples: 5000


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

model = CRNN(num_classes=len(char_to_idx) + 1).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0003)
criterion = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=False)

Using device: cuda


In [13]:
checkpoint = torch.load("crnn_word_reader_final.pth")

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

start_epoch = checkpoint['epoch']
for epoch in range(50):
    model.train()
    running_loss = 0.0

    for images, targets, target_lengths in train_loader:
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        preds = model(images)           # [B,T,C]
        preds = preds.permute(1, 0, 2)  # [T,B,C]
        preds = preds.log_softmax(2)

        input_lengths = torch.full(
            (preds.size(1),),
            preds.size(0),
            dtype=torch.long
        )

        loss = criterion(
            preds,
            targets,
            input_lengths,
            target_lengths
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f'Epoch {epoch+1}/50 | Loss: {avg_loss:.4f}')
    torch.save(model.state_dict(), 'crnn_word_reader_final.pth')
    print('Saved model to crnn_word_reader_final.pth')

Epoch 1/50 | Loss: 5.3709
Saved model to crnn_word_reader_final.pth
Epoch 2/50 | Loss: 3.4090
Saved model to crnn_word_reader_final.pth
Epoch 3/50 | Loss: 3.0302
Saved model to crnn_word_reader_final.pth
Epoch 4/50 | Loss: 2.8243
Saved model to crnn_word_reader_final.pth
Epoch 5/50 | Loss: 2.6473
Saved model to crnn_word_reader_final.pth
Epoch 6/50 | Loss: 2.4886
Saved model to crnn_word_reader_final.pth
Epoch 7/50 | Loss: 2.3312
Saved model to crnn_word_reader_final.pth
Epoch 8/50 | Loss: 2.1967
Saved model to crnn_word_reader_final.pth
Epoch 9/50 | Loss: 2.0719
Saved model to crnn_word_reader_final.pth
Epoch 10/50 | Loss: 1.9330
Saved model to crnn_word_reader_final.pth
Epoch 11/50 | Loss: 1.8201
Saved model to crnn_word_reader_final.pth
Epoch 12/50 | Loss: 1.7207
Saved model to crnn_word_reader_final.pth
Epoch 13/50 | Loss: 1.6136
Saved model to crnn_word_reader_final.pth
Epoch 14/50 | Loss: 1.5193
Saved model to crnn_word_reader_final.pth
Epoch 15/50 | Loss: 1.4322
Saved model to c

In [ ]:
torch.save(model.state_dict(), 'crnn_word_reader_final.pth')
print('Saved model to crnn_word_reader_final.pth')

In [14]:
def decode_prediction(output):
    output = output.argmax(2)
    output = output[:, 0]

    result = []
    prev = -1

    for p in output:
        p = p.item()

        if p != prev and p != BLANK_IDX:
            result.append(idx_to_char.get(p, ''))

        prev = p

    return ''.join(result)

model.eval()
with torch.no_grad():
    images, targets, target_lengths = next(iter(train_loader))
    images = images.to(device)

    out = model(images)
    out = out.permute(1, 0, 2)

    print('Prediction:', decode_prediction(out))

Prediction: attack
